# RT-DETR Football Fine-Tuning

Bu notebook, **RT-DETR-L** modelini futbol veri seti ile fine-tune eder.\
Veri seti: `football-players-detection-1` (Roboflow) — 4 sınıf: `ball`, `goalkeeper`, `player`, `referee`

---
**Gereksinimler:**
```
pip install ultralytics
```

> **Not:** Google Colab'da çalıştırıyorsan sağ üstten GPU runtime seç.

## 1. Ortam Kontrolü

In [9]:
import torch
from ultralytics import RTDETR

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch version : 2.5.1+cu121
CUDA available  : True
GPU             : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM            : 4.0 GB


## 2. Veri Seti Yolu Ayarla

> **Not:** Orijinal `data.yaml` içindeki `../train/images` path'i hatalı (Roboflow'un çıktısı):  
> - Klasörler `football-players-detection-1/` **içinde** ama `../` ile bir üst dizine bakıyor.  
> - Bu yüzden düzeltilmiş absolute path'li `data_abs.yaml` kullanıyoruz.

In [10]:
import os

PROJECT_ROOT = os.path.abspath('..')  # notebook training/ altında
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')

# Absolute path'li yaml — train/val/test yolları doğru absolute path içeriyor
DATA_YAML = os.path.join(
    PROJECT_ROOT, 'training', 'football-players-detection-1', 'data_abs.yaml'
)

assert os.path.exists(DATA_YAML), f'data_abs.yaml bulunamadı: {DATA_YAML}'
print(f'data.yaml  : {DATA_YAML}')
print(f'models dir : {MODELS_DIR}')

data.yaml  : /home/samet/Projects/football_analysis/training/football-players-detection-1/data_abs.yaml
models dir : /home/samet/Projects/football_analysis/models


## 3. Model Yükle

In [11]:
# rtdetr-l.pt : büyük model, daha iyi mAP ama daha yavaş
# rtdetr-x.pt : en büyük model
# rtdetr-s.pt : küçük model, hızlı (test için)
model = RTDETR('rtdetr-l.pt')  # İlk çalıştırmada otomatik indirir
print(model.info())

rt-detr-l summary: 449 layers, 32,970,476 parameters, 0 gradients, 108.3 GFLOPs
(449, 32970476, 0, 108.3437056)


## 4. Fine-Tune

### Eğitim Parametreleri

| Parametre | Değer | Açıklama |
|---|---|---|
| `epochs` | 50 | Epoch sayısı — GPU'ya göre artırılabilir |
| `imgsz` | 1280 | Yüksek çözünürlük (toplar için kritik) |
| `batch` | 4 | VRAM'e göre ayarla (8GB → 4, 16GB → 8) |
| `optimizer` | AdamW | RT-DETR için önerilen |
| `lr0` | 0.0001 | Düşük LR (pretrained ağırlıkları korumak için) |
| `patience` | 15 | Early stopping |
| `mosaic` | 1.0 | Mozaik augmentation |
| `fliplr` | 0.5 | Yatay flip (sahada geçerli) |
| `flipud` | 0.0 | Dikey flip (futbolda anlamsız, kapalı) |
| `degrees` | 0.0 | Rotasyon (kamera sabit, kapalı) |

> ⚠️ **`augment` parametresi burada kullanılmaz!**  
> `augment=True` sadece `model.predict()` sırasında Test-Time Augmentation (TTA) açmak için kullanılır.  
> Eğitim augmentation'ı `mosaic`, `fliplr`, `flipud` gibi ayrı parametrelerle ayarlanır.

In [13]:
results = model.train(
    data      = DATA_YAML,
    epochs    = 20,
    imgsz     = 1280,
    batch     = 4,        # VRAM'e göre ayarla (8GB→4, 16GB→8)
    optimizer = 'AdamW',
    lr0       = 1e-4,
    lrf       = 0.01,     # Final LR = lr0 * lrf
    patience  = 15,       # Early stopping
    # Augmentation — augment=True YAZILMAZ (predict-time TTA parametresi!)
    mosaic    = 1.0,      # Mozaik (4 görüntüyü birleştir)
    fliplr    = 0.5,      # %50 yatay flip
    flipud    = 0.0,      # Dikey flip kapalı
    degrees   = 0.0,      # Rotasyon kapalı
    project   = os.path.join(PROJECT_ROOT, 'training', 'runs'),
    name      = 'rtdetr_football',
    exist_ok  = True,
    device    = 0 if torch.cuda.is_available() else 'cpu',
    verbose   = True,
)

New https://pypi.org/project/ultralytics/8.4.30 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.12.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3770MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/samet/Projects/football_analysis/training/football-players-detection-1/data_abs.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train,

Exception in thread Thread-30 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/samet/Projects/football_analysis/cv_env/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 59, in _pin_memory_loop
    do_one_step()
  File "/home/samet/Projects/football_analysis/cv_env/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 35, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/samet/Projects/football_analysis/cv_env/lib/python3.12/site-packages/torch/multiprocessing/reductions.py", line 541, in rebuild_s

optimizer: AdamW(lr=0.0001, momentum=0.937) with parameter groups 143 weight(decay=0.0), 206 weight(decay=0.0005), 226 bias(decay=0.0)
: 0% ──────────── 0/612  6.2s

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
: 0% ──────────── 0/612  0.1s


OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 3.25 MiB is free. Including non-PyTorch memory, this process has 3.62 GiB memory in use. Of the allocated memory 3.47 GiB is allocated by PyTorch, and 30.91 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 5. En İyi Modeli `models/` Klasörüne Kopyala

In [ ]:
import shutil

best_pt   = results.save_dir / 'weights' / 'best.pt'
output_pt = os.path.join(MODELS_DIR, 'rtdetr_best.pt')

os.makedirs(MODELS_DIR, exist_ok=True)
shutil.copy(best_pt, output_pt)
print(f'Model kaydedildi → {output_pt}')

## 6. Validation Metrikleri

In [ ]:
val_results = model.val(data=DATA_YAML, imgsz=1280, verbose=True)

print('\n─── Validation Sonuçları ───')
print(f'mAP50      : {val_results.box.map50:.4f}')
print(f'mAP50-95   : {val_results.box.map:.4f}')

class_names = ['ball', 'goalkeeper', 'player', 'referee']
if hasattr(val_results.box, 'ap_class_index') and val_results.box.ap_class_index is not None:
    print('\nSınıf bazlı AP50:')
    for i, cls_idx in enumerate(val_results.box.ap_class_index):
        name = class_names[cls_idx] if cls_idx < len(class_names) else f'cls_{cls_idx}'
        print(f'  {name:<12}: {val_results.box.ap50[i]:.4f}')

## 7. YOLO ile Karşılaştırma

Eğitim tamamlandıktan sonra terminal'de şunu çalıştır:

```bash
source cv_env/bin/activate
python model_comparison.py \
    --yolo   models/best.pt \
    --rtdetr models/rtdetr_best.pt \
    --data   training/football-players-detection-1/data_abs.yaml \
    --imgsz  1280
```

Sonuçlar `output_videos/comparison_results.csv` dosyasına kaydedilir.

## 8. (Opsiyonel) RT-DETR ile Video Test

In [ ]:
from ultralytics import RTDETR

# Fine-tune edilmiş modeli yükle
rtdetr_model = RTDETR(output_pt)

VIDEO_PATH = os.path.join(PROJECT_ROOT, 'input_videos', 'test (20).mp4')

if os.path.exists(VIDEO_PATH):
    results = rtdetr_model.predict(
        source  = VIDEO_PATH,
        save    = True,
        conf    = 0.3,
        imgsz   = 1280,
        # augment=True  ← buraya YAZILABİLİR (predict-time TTA, isteğe bağlı)
        project = os.path.join(PROJECT_ROOT, 'output_videos'),
        name    = 'rtdetr_test'
    )
    print('Video kaydedildi → output_videos/rtdetr_test/')
else:
    print(f'Video bulunamadı: {VIDEO_PATH}')